In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

DATA_FILE = "Water Quality.csv"
LOCATION_COLUMN = "MonitoringLocationID"
PARAMETER_COLUMN = "CharacteristicName"
VALUE_COLUMN = "ResultValue"
DATE_COLUMN = "ActivityStartDate"
OUTPUT_DIR = "task2_outputs"
MIN_PARAMETER_COVERAGE = 0.20

os.makedirs(OUTPUT_DIR, exist_ok=True)
df = pd.read_csv(DATA_FILE, low_memory=False)

required = [LOCATION_COLUMN, PARAMETER_COLUMN, VALUE_COLUMN]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}\nAvailable columns: {list(df.columns)}")

df[VALUE_COLUMN] = pd.to_numeric(df[VALUE_COLUMN], errors="coerce")
df = df.dropna(subset=[LOCATION_COLUMN, PARAMETER_COLUMN, VALUE_COLUMN]).copy()

# Each row in the pivot represents a location. If a date column exists, use
# location + date so that measurements from different sampling dates stay apart.
index_columns = [LOCATION_COLUMN]
if DATE_COLUMN in df.columns:
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
    if df[DATE_COLUMN].notna().any():
        index_columns.append(DATE_COLUMN)

# Median aggregation handles repeated measurements of the same parameter in a
# location/date record. The median is less sensitive to extreme values.
wide = df.pivot_table(
    index=index_columns,
    columns=PARAMETER_COLUMN,
    values=VALUE_COLUMN,
    aggfunc="median"
)

# Remove parameters that are too sparse or constant.
minimum_non_missing = max(3, int(len(wide) * MIN_PARAMETER_COVERAGE))
wide = wide.dropna(axis=1, thresh=minimum_non_missing)
wide = wide.loc[:, wide.nunique(dropna=True) > 1]

if wide.shape[1] < 2:
    raise ValueError(
        "PCA needs at least two varying parameters with enough observations. "
        "Try lowering MIN_PARAMETER_COVERAGE or check the dataset's missing values."
    )

# Median-impute remaining missing measurements, then standardize all features.
wide = wide.apply(pd.to_numeric, errors="coerce")
wide = wide.fillna(wide.median())
if len(wide) < 3:
    raise ValueError("PCA needs at least three usable rows after pivoting.")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(wide)
pca = PCA()
scores = pca.fit_transform(X_scaled)

ratio = pca.explained_variance_ratio_
cumulative = np.cumsum(ratio)
variance = pd.DataFrame({
    "Component": [f"PC{i+1}" for i in range(len(ratio))],
    "Eigenvalue": pca.explained_variance_,
    "Explained_variance_percent": ratio * 100,
    "Cumulative_explained_percent": cumulative * 100
})
variance.to_csv(os.path.join(OUTPUT_DIR, "pca_explained_variance.csv"), index=False)

target_variance = 0.90
n_target = int(np.argmax(cumulative >= target_variance) + 1) if cumulative[-1] >= target_variance else len(cumulative)
print("PCA input shape:", wide.shape)
print("\nExplained variance:")
print(variance.round(4).to_string(index=False))
print(f"\nComponents needed for 90% cumulative variance (if achievable): {n_target}")

scores_df = pd.DataFrame(scores, index=wide.index, columns=[f"PC{i+1}" for i in range(scores.shape[1])])
scores_df.to_csv(os.path.join(OUTPUT_DIR, "pca_transformed_features.csv"))

# Scree plot with individual and cumulative explained variance.
components = np.arange(1, len(ratio) + 1)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(components, ratio * 100, marker="o", label="Individual variance")
ax.set_xlabel("Principal component")
ax.set_ylabel("Explained variance (%)")
ax.set_xticks(components)
ax.grid(alpha=0.3)
ax2 = ax.twinx()
ax2.plot(components, cumulative * 100, marker="s", linestyle="--", label="Cumulative variance")
ax2.set_ylabel("Cumulative explained variance (%)")
plt.title("AquaInsight: PCA Scree Plot")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "pca_scree_plot.png"), dpi=250)
plt.close(fig)

# Score plot: each point is one location or location/date sample.
plt.figure(figsize=(8, 6))
plt.scatter(scores[:, 0], scores[:, 1], alpha=0.55, s=20)
plt.xlabel(f"PC1 ({ratio[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({ratio[1]*100:.1f}% variance)")
plt.title("PCA Score Plot: Water Monitoring Samples")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pca_score_plot.png"), dpi=250)
plt.close()

# Biplot of scores and approximate feature-loading directions.
plt.figure(figsize=(10, 8))
plt.scatter(scores[:, 0], scores[:, 1], alpha=0.30, s=15)
loadings = pca.components_.T[:, :2]
span = min(np.ptp(scores[:, 0]) or 1, np.ptp(scores[:, 1]) or 1)
arrow_scale = 0.35 * span
for i, feature in enumerate(wide.columns):
    dx = loadings[i, 0] * arrow_scale
    dy = loadings[i, 1] * arrow_scale
    plt.arrow(0, 0, dx, dy, color="red", alpha=0.7,
              head_width=0.04 * arrow_scale, length_includes_head=True)
    plt.text(dx * 1.08, dy * 1.08, str(feature), fontsize=7, color="darkred")
plt.axhline(0, color="grey", linewidth=0.7)
plt.axvline(0, color="grey", linewidth=0.7)
plt.xlabel(f"PC1 ({ratio[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({ratio[1]*100:.1f}% variance)")
plt.title("PCA Biplot (PC1 vs PC2)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pca_biplot.png"), dpi=250)
plt.close()

print(f"\nAll Task 2 outputs saved in: {OUTPUT_DIR}")
print(
    "Interpretation: the scree plot shows how much variation each component "
    "captures. Similar loading-arrow directions suggest positively associated "
    "feature patterns. PCA is unsupervised and does not guarantee improved "
    "prediction accuracy."
)


PCA input shape: (7283, 41)

Explained variance:
Component  Eigenvalue  Explained_variance_percent  Cumulative_explained_percent
      PC1     10.1603                     24.7777                       24.7777
      PC2      4.4217                     10.7832                       35.5609
      PC3      2.9633                      7.2266                       42.7875
      PC4      2.8143                      6.8633                       49.6508
      PC5      2.4487                      5.9716                       55.6224
      PC6      2.0029                      4.8844                       60.5068
      PC7      1.7148                      4.1818                       64.6886
      PC8      1.2597                      3.0719                       67.7605
      PC9      1.0748                      2.6211                       70.3816
     PC10      1.0173                      2.4808                       72.8624
     PC11      0.9318                      2.2724                      